# ⚙️ MVP-C：MCP x Multi-Agent 實戰流程展示
### 技術：LangGraph + OpenAI + Tool Calling

這個 Notebook 展示了一個真實的「多代理人工作流」。研究員與撰稿員會進行對話，並模擬從 MCP Server 獲取外部工具的結果。

In [ ]:
!pip install langgraph langchain-openai

In [ ]:
import os
from google.colab import userdata
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY') if 'google.colab' in str(get_ipython()) else "YOUR_API_KEY"

class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], "對話歷史"]

model = ChatOpenAI(model="gpt-4-turbo-preview", temperature=0)

def researcher_node(state: AgentState):
    # 模擬研究員與 LLM 互動，這一步可以串接真實 MCP 工具
    msg = model.invoke(state['messages'] + [HumanMessage(content="作為研究員，請列出這份專案的三個關鍵技術指標。")])
    return {"messages": [msg]}

def writer_node(state: AgentState):
    # 撰稿員根據研究結果生成報告
    msg = model.invoke(state['messages'] + [HumanMessage(content="作為撰稿員，請將研究結果整理成一份專業的商業報告。")])
    return {"messages": [msg]}

workflow = StateGraph(AgentState)
workflow.add_node("researcher", researcher_node)
workflow.add_node("writer", writer_node)
workflow.set_entry_point("researcher")
workflow.add_edge("researcher", "writer")
workflow.add_edge("writer", END)

graph = workflow.compile()
print("多代理人實戰工作流編譯完成")

In [ ]:
# 執行工作流並展示真實對話
inputs = {"messages": [HumanMessage(content="專案目標：針對吉永建設導入 AI 雙引擎套件。")]}
for output in graph.stream(inputs):
    for key, value in output.items():
        print(f"\n--- {key} --- ")
        print(value["messages"][-1].content)